# 练习 2：用于 TinyML 唤醒词检测的计算图实现

## MAIE 5532：机器学习系统 - 第 2 周

### 学习目标：
• 理解反向模式自动微分中的计算图构建
• 实现前向传播并进行策略性的中间值存储
• 实现反向传播以完成完整梯度计算
• 应用嵌入式系统中的内存管理技术
• 将理论与实际唤醒词检测系统联系起来

### 🎯 什么是计算图？

计算图是反向模式自动微分的基础数据结构。可以将它看作是一个“记住你计算每一步”的配方：

- **节点**：表示数值（输入、参数、中间结果、输出）
- **边**：表示将数值转换的操作
- **前向传播**：按照配方进行计算，并保存中间材料
- **反向传播**：反向追踪配方，找出每个材料如何影响最终结果

### 为什么这对唤醒词检测很重要？

每当智能音箱识别“Hey Siri”或“OK Google”时，它都在使用由计算图训练得到的神经网络。这些图中计算出的梯度使系统能够从海量语音样本中学习。

### 实际应用影响：
- **隐私保护**：端侧学习意味着你的语音不会离开设备
- **个性化**：模型会适应你的特定语音模式
- **效率**：针对微控制器部署进行了优化
- **联邦学习**：无需共享数据即可为全局模型改进做贡献

In [1]:
# 计算图实现所需的基础导入
import math
import numpy as np
from IPython.core.display import HTML

# 用于清晰展示结果的工具函数
def show(title, *pairs):
    """
    以结构化方式展示结果的格式化打印函数
    让输出更易读、更专业
    """
    print(title)
    for k, v in pairs:
        print(f"  {k}: {v}")

print("🚀 练习 2：用于唤醒词检测的计算图")
print("=" * 60)
print()
print("✅ 导入完成成功！")
print()
print("📝 代码说明：")
print("  • numpy：用于矩阵运算的高效数值计算库")
print("  • math：基础数学函数")
print("  • show()：用于清晰结果展示的自定义格式化函数")
print()
print("🧠 计算图概念：")
print("  计算图会记录神经网络中的每一步运算")
print("  它就像一份详细的食谱，记住了每一步烹饪过程")
print("  前向传播：按步骤执行配方，反向传播：反向推导每一步")

🚀 练习 2：用于唤醒词检测的计算图

✅ 导入完成成功！

📝 代码说明：
  • numpy：用于矩阵运算的高效数值计算库
  • math：基础数学函数
  • show()：用于清晰结果展示的自定义格式化函数

🧠 计算图概念：
  计算图会记录神经网络中的每一步运算
  它就像一份详细的食谱，记住了每一步烹饪过程
  前向传播：按步骤执行配方，反向传播：反向推导每一步


## 🏗️ 构建计算图基础设施

### 节点设计理念

我们的 `ComputationNode` 类是计算图的构建基石。每个节点都需要存储：

1. **值**：前向传播中的实际数值结果
2. **梯度**：该节点对应的最终输出变化量（反向传播）
3. **依赖关系**：该节点依赖哪些节点（图结构）
4. **反向函数**：如何为该特定操作计算梯度

### 内存管理策略

对于嵌入式系统，我们必须非常小心地管理内存使用：
- **预分配**：在编译时就知道内存需求
- **策略性存储**：只保留反向传播绝对必要的信息
- **内存池**：尽可能复用内存位置
- **混合精度**：尽量使用 16 位，在必要时使用 32 位

### 反向模式的神奇之处

与前向模式（一次只计算一个输入的导数）不同，反向模式：
- ✅ 在一次反向传播中计算所有参数梯度
- ✅ 对神经网络非常高效（参数数以百万计，但仅有一个损失值）
- ⚠️ 需要存储中间值（内存成本）
- 🎯 非常适合训练，但对仅推理的嵌入式系统较具挑战

In [2]:
class ComputationNode:
    """
    计算图中的一个节点，用于反向模式自动微分
    
    这就像一个“智能容器”，它保存着：
    - 前向传播中计算得到的值
    - 反向传播中计算得到的梯度
    - 父节点的引用（依赖关系）
    - 计算梯度的函数（反向函数）
    
    可以把它想象成一个食谱步骤，既记住了结果，
    也记住了如何反向撤销该步骤来计算梯度。
    """
    
    def __init__(self, value, name="", requires_grad=True):
        """
        创建一个新的计算节点
        
        参数：
            value: 数值（标量、向量或矩阵）
            name: 用于调试的人类可读标识符
            requires_grad: 是否需要计算梯度
        """
        # 以 NumPy 数组存储值，以保证操作一致性
        self.value = np.array(value, dtype=np.float32)
        
        # 初始化梯度为零（将在反向传播过程中填充）
        self.gradient = np.zeros_like(self.value)
        
        # 调试和图可视化所需的元数据
        self.name = name
        self.requires_grad = requires_grad
        
        # 图结构信息
        self.backward_fn = None  # 计算梯度的函数
        self.inputs = []        # 图中的父节点
        
        print(f"📦 已创建节点 '{self.name}'：")
        print(f"    形状: {self.value.shape}")
        print(f"    值: {self.value}")
        print(f"    是否需要梯度: {requires_grad}")
    
    def __repr__(self):
        """便于调试的字符串表示"""
        return f"Node('{self.name}', shape={self.value.shape}, grad_norm={np.linalg.norm(self.gradient):.6f})"

print("✅ ComputationNode 类已实现！")
print()
print("🔍 我们刚刚构建了什么：")
print("  • 一个用于保存值和梯度的‘智能容器’")
print("  • 用 NumPy 数组进行内存高效存储")
print("  • 通过输入引用追踪图结构")
print("  • 通过名称和表示形式支持调试")
print()
print("💡 关键思想：")
print("  每个节点都像一个食谱步骤，既记住了结果")
print("  也记住了如何反向计算梯度以实现梯度流动")

✅ ComputationNode 类已实现！

🔍 我们刚刚构建了什么：
  • 一个用于保存值和梯度的‘智能容器’
  • 用 NumPy 数组进行内存高效存储
  • 通过输入引用追踪图结构
  • 通过名称和表示形式支持调试

💡 关键思想：
  每个节点都像一个食谱步骤，既记住了结果
  也记住了如何反向计算梯度以实现梯度流动


## 🧮 实现神经网络运算

### 线性层：神经网络的主力

线性（全连接）层是基础模块：**output = input @ weight.T + bias**

### （@ 符号是 Python 中的矩阵乘法运算符）

**前向传播数学公式：**
- 矩阵乘法：利用学习到的权重组合输入特征
- 偏置加法：移动决策边界
- 结果：将特征转换为下一层的表示

**反向传播数学公式（链式法则）：**
- ∂Loss/∂input = ∂Loss/∂output @ weight
- ∂Loss/∂weight = ∂Loss/∂output.T @ input
- ∂Loss/∂bias = sum(∂Loss/∂output)

### 为什么采用这种实现策略？

1. **模块化**：每个运算都是自包含的
2. **可组合性**：运算可以按顺序串联起来
3. **自动梯度**：反向函数会自动处理链式法则
4. **内存效率**：只存储反向传播真正需要的信息

### 现实世界联系

这正是 PyTorch 和 TensorFlow 实现其操作的方式！我们的实现展示了核心原理，而没有引入生产级框架的复杂性。

In [3]:
def linear_layer(input_node, weight_node, bias_node, name="linear"):
    """
    线性（全连接）层：output = input @ weight.T + bias
    
    这是神经网络的基本构建块。
    我们同时计算前向传播，并设置反向传播函数。
    
    数学运算：
    1. 矩阵乘法：input @ weight.T
    2. 偏置相加：result + bias
    3. 存储反向传播所需的一切信息
    
    参数：
        input_node: 输入特征（形状：[input_dim]）
        weight_node: 权重矩阵（形状：[output_dim, input_dim]）
        bias_node: 偏置向量（形状：[output_dim]）
        name: 该层的人类可读名称
    
    返回：
        output_node: 线性变换结果
    """
    print(f"\n🔄 计算 {name} 层")
    print(f"    输入形状: {input_node.value.shape}")
    print(f"    权重形状: {weight_node.value.shape}")
    print(f"    偏置形状: {bias_node.value.shape}")
    
    # === 前向传播计算 ===
    # 第 1 步：矩阵乘法 (input @ weight.T)
    matmul_result = input_node.value @ weight_node.value.T
    print(f"    矩阵乘法后: {matmul_result.shape}")
    
    # 第 2 步：加偏置
    linear_output = matmul_result + bias_node.value
    print(f"    加偏置后: {linear_output.shape}")
    print(f"    输出值: {linear_output}")
    
    # 创建输出节点
    output_node = ComputationNode(linear_output, name=f"{name}_output")
    
    # 存储输入引用以进行反向传播
    output_node.inputs = [input_node, weight_node, bias_node]
    
    def backward():
        """
        线性层的反向传播 - 实现链式法则
        
        给定反向流回的梯度（∂Loss/∂output），计算：
        - ∂Loss/∂input = ∂Loss/∂output @ weight
        - ∂Loss/∂weight = ∂Loss/∂output.T @ input
        - ∂Loss/∂bias = ∂Loss/∂output（若为批量数据则求和）
        
        这就是链式法则的实际应用！
        """
        print(f"\n⬅️ {name} 的反向传播")
        print(f"    传入梯度形状: {output_node.gradient.shape}")
        print(f"    传入梯度: {output_node.gradient}")
        
        # 对输入的梯度：∂Loss/∂input = ∂Loss/∂output @ weight
        if input_node.requires_grad:
            input_grad = output_node.gradient @ weight_node.value
            input_node.gradient += input_grad
            print(f"    ∂Loss/∂input: {input_grad}")
        
        # 对权重的梯度：∂Loss/∂weight = ∂Loss/∂output.T @ input
        if weight_node.requires_grad:
            # 重塑以进行正确的矩阵乘法
            grad_reshaped = output_node.gradient.reshape(-1, 1) if output_node.gradient.ndim == 1 else output_node.gradient
            input_reshaped = input_node.value.reshape(1, -1) if input_node.value.ndim == 1 else input_node.value
            
            weight_grad = grad_reshaped @ input_reshaped
            weight_node.gradient += weight_grad
            print(f"    ∂Loss/∂weight 形状: {weight_grad.shape}")
            print(f"    ∂Loss/∂weight: {weight_grad}")
        
        # 对偏置的梯度：∂Loss/∂bias = ∂Loss/∂output
        if bias_node.requires_grad:
            bias_grad = output_node.gradient.copy()
            bias_node.gradient += bias_grad
            print(f"    ∂Loss/∂bias: {bias_grad}")
    
    # 将反向函数附加到输出节点
    output_node.backward_fn = backward
    return output_node

print("✅ 线性层实现完成！")
print()
print("🔍 这个函数做了什么：")
print("  • 前向：计算 output = input @ weight.T + bias")
print("  • 反向：使用链式法则设置梯度计算")
print("  • 内存：存储输入引用（不是复制值！）")
print("  • 模块化：可以与其他运算链接起来")
print()
print("💡 链式法则实现：")
print("  反向函数会自动应用微积分规则")
print("  不需要手动计算导数！")

✅ 线性层实现完成！

🔍 这个函数做了什么：
  • 前向：计算 output = input @ weight.T + bias
  • 反向：使用链式法则设置梯度计算
  • 内存：存储输入引用（不是复制值！）
  • 模块化：可以与其他运算链接起来

💡 链式法则实现：
  反向函数会自动应用微积分规则
  不需要手动计算导数！


## 🎛️ 激活函数：非线性与梯度流动

### Tanh 激活：平滑的非线性

**数学性质：**
- **取值范围**：(-1, 1) - 保持数值有界
- **导数**：1 - tanh²(x) - 可直接从前向传播值计算
- **零中心特性**：有助于深层网络中的梯度流动

**为什么 Tanh 适合隐藏层？**
- 平滑梯度（没有突然跳变）
- 零中心输出有助于下一层训练
- 饱和较平滑（极端情况下梯度接近 0）

### ReLU：深度学习的革命

**数学性质：**
- **函数**：max(0, x)
- **导数**：若 x > 0，则为 1；否则为 0
- **计算成本**：极低（只是简单比较）

**为什么 ReLU 改变了一切：**
- 解决了深层网络中的梯度消失问题
- 稀疏激活（许多神经元输出为 0）
- 对嵌入式系统计算效率极高

### 梯度消失问题
这类梯度是通过不断相乘多个小数得到的（具体来说，是每层激活函数的导数，例如 sigmoid 或 tanh，它们通常都在 0 和 1 之间）。

就像不断乘以分数（例如 0.5 × 0.5 × 0.5 ...）一样，这种链式乘法会导致梯度随着反向传播逐层指数衰减，最终变得几乎为 0，早期层就停止学习。

ReLU 会把这些小分数变成 1（当它们大于 0 时）。

### Softmax：概率分布

**数学性质：**
- **函数**：exp(xi) / sum(exp(x))
- **输出**：有效的概率分布（总和为 1）
- **导数**：复杂但与交叉熵损失结合时非常优雅

**非常适合分类：**
- 将 logits 转换为概率
- 放大不同类别之间的差异
- 与交叉熵损失自然结合

In [4]:
def tanh_activation(input_node, name="tanh"):
    """
    双曲正切激活函数
    
    前向：tanh(x) = (e^x - e^(-x)) / (e^x + e^(-x))
    反向：d/dx[tanh(x)] = 1 - tanh²(x)
    
    关键特性：导数可以直接从前向传播结果中计算出来！
    这样可以在反向传播中节省计算量和内存。
    """
    print(f"\n🎛️ 计算 {name} 激活")
    print(f"    输入值: {input_node.value}")
    
    # 前向传播：计算 tanh
    tanh_output = np.tanh(input_node.value)
    output_node = ComputationNode(tanh_output, name=f"{name}_output")
    output_node.inputs = [input_node]
    
    print(f"    输出值: {output_node.value}")
    print(f"    输出范围: [{np.min(output_node.value):.3f}, {np.max(output_node.value):.3f}]")
    
    def backward():
        """
        tanh 激活的反向传播
        
        关键思想：tanh'(x) = 1 - tanh²(x)
        我们可以直接利用前向传播结果（output_node.value）来计算它！
        """
        print(f"\n⬅️ {name} 的反向传播")
        
        if input_node.requires_grad:
            # 导数：1 - tanh²(x)
            tanh_derivative = 1 - output_node.value ** 2
            
            # 应用链式法则：incoming_gradient * local_derivative
            input_grad = output_node.gradient * tanh_derivative
            input_node.gradient += input_grad
            
            print(f"    tanh 导数 (1-tanh²): {tanh_derivative}")
            print(f"    输入梯度: {input_grad}")
    
    output_node.backward_fn = backward
    return output_node

def softmax_activation(input_node, name="softmax"):
    """
    用于分类的 Softmax 激活
    
    前向：softmax(x)_i = exp(x_i) / sum(exp(x_j))
    特性：
    - 输出总和为 1（有效概率分布）
    - 放大不同类别之间的差异
    - 通过减去最大值实现数值稳定
    """
    print(f"\n🎯 计算 {name} 激活")
    print(f"    输入 logits: {input_node.value}")
    
    # 数值稳定性：减去最大值以防止溢出
    max_val = np.max(input_node.value)
    stable_input = input_node.value - max_val
    print(f"    稳定化后: {stable_input}")
    
    # 计算 softmax
    exp_values = np.exp(stable_input)
    softmax_output = exp_values / np.sum(exp_values)
    
    output_node = ComputationNode(softmax_output, name=f"{name}_output")
    output_node.inputs = [input_node]
    
    print(f"    输出概率: {output_node.value}")
    print(f"    概率和: {np.sum(output_node.value):.6f}（应为 1.0）")
    
    def backward():
        """
        softmax 的反向传播
        
        梯度公式较复杂，但与交叉熵损失结合时，
        会简化为：softmax_output - target
        """
        print(f"\n⬅️ {name} 的反向传播")
        
        if input_node.requires_grad:
            # Softmax Jacobian 矩阵计算
            s = output_node.value.reshape(-1, 1)
            jacobian = np.diagflat(s) - np.dot(s, s.T)
            
            # 应用链式法则
            input_grad = jacobian @ output_node.gradient.reshape(-1, 1)
            input_node.gradient += input_grad.flatten()
            
            print(f"    Jacobian 形状: {jacobian.shape}")
            print(f"    输入梯度: {input_node.gradient}")
    
    output_node.backward_fn = backward
    return output_node

print("✅ 激活函数已实现！")
print()
print("🎛️ 激活函数特性：")
print("  • tanh：平滑、零中心、范围在 (-1, 1) 内")
print("  • softmax：概率分布，适合分类任务")
print("  • 导数可直接从前向传播值高效计算")
print()
print("🧠 深度学习洞察：")
print("  激活函数提供非线性，使神经网络能够学习")
print("  复杂模式和决策边界")

✅ 激活函数已实现！

🎛️ 激活函数特性：
  • tanh：平滑、零中心、范围在 (-1, 1) 内
  • softmax：概率分布，适合分类任务
  • 导数可直接从前向传播值高效计算

🧠 深度学习洞察：
  激活函数提供非线性，使神经网络能够学习
  复杂模式和决策边界


## 🎯 交叉熵损失：用于分类的损失函数

### 为什么使用交叉熵损失？

交叉熵损失是分类任务的黄金标准，因为：

1. **概率解释**：衡量预测概率与真实分布之间的差距
2. **梯度特性**：当预测错误时，能够提供强梯度
3. **数学优雅**：与 softmax 激活函数结合得非常自然
4. **凸性**：对线性模型具有良好的优化性质（没有局部极小值）

### 数学之美

**前向**：Loss = -∑(target_i × log(prediction_i))
**反向**（配合 softmax）：∂Loss/∂logits = predictions - targets

这种简化正是 softmax + cross-entropy 在深度学习中无处不在的原因！

### 数值稳定性考虑

- **Log(0) 问题**：加入很小的 epsilon 以避免 -∞
- **溢出防护**：已由 softmax 稳定化处理
- **梯度裁剪**：在病理情况下防止梯度爆炸

### 唤醒词检测场景

对于唤醒词检测：
- **类别 0**：检测到唤醒词 → target = [1, 0]
- **类别 1**：未检测到唤醒词 → target = [0, 1]
- **训练目标**：最小化交叉熵，以提高分类准确率

In [5]:
def cross_entropy_loss(predictions_node, target_node, name="cross_entropy"):
    """
    用于分类的交叉熵损失
    
    前向：Loss = -sum(target * log(predictions))
    
    这是分类任务的标准损失函数。
    它会对“自信但错误”的预测进行重罚，并提供
    强梯度以促进学习。
    
    参数：
        predictions_node: Softmax 概率向量 [num_classes]
        target_node: One-hot 编码的真实类别 [num_classes]
        
    返回：
        loss_node: 标量损失值
    """
    print(f"\n🎯 计算 {name} 损失")
    print(f"    预测值: {predictions_node.value}")
    print(f"    目标值: {target_node.value}")
    
    # 数值稳定性：避免 log(0)
    epsilon = 1e-15
    safe_predictions = np.clip(predictions_node.value, epsilon, 1 - epsilon)
    print(f"    安全预测值（裁剪后）: {safe_predictions}")
    
    # 交叉熵计算：-sum(target * log(predictions))
    log_predictions = np.log(safe_predictions)
    loss_terms = target_node.value * log_predictions
    loss_value = -np.sum(loss_terms)
    
    print(f"    对数预测值: {log_predictions}")
    print(f"    损失项: {loss_terms}")
    print(f"    最终损失: {loss_value}")
    
    # 创建标量损失节点
    loss_node = ComputationNode(loss_value, name=f"{name}_output")
    loss_node.inputs = [predictions_node, target_node]
    
    def backward():
        """
        交叉熵损失的反向传播
        
        梯度：∂Loss/∂predictions = -target / predictions
        
        当与 softmax 结合时，它会进一步简化为：
        predictions - target（非常优雅）
        """
        print(f"\n⬅️ {name} 的反向传播")
        
        if predictions_node.requires_grad:
            # 对预测值的梯度
            pred_grad = -target_node.value / safe_predictions
            predictions_node.gradient += pred_grad
            
            print(f"    对预测值的梯度: {pred_grad}")
            print(f"    梯度解释:")
            for i, (pred, target, grad) in enumerate(zip(predictions_node.value, target_node.value, pred_grad)):
                if target > 0:  # 真类别
                    print(f"      类别 {i}（TRUE）：pred={pred:.4f}, grad={grad:.4f}")
                    print(f"        → {'强' if abs(grad) > 1 else '弱'}梯度推动概率上升")
                else:  # 假类别
                    print(f"      类别 {i}（FALSE）：pred={pred:.4f}, grad={grad:.4f}")
                    print(f"        → {'强' if abs(grad) > 1 else '弱'}梯度推动概率下降")
    
    loss_node.backward_fn = backward
    return loss_node

print("✅ 交叉熵损失已实现！")
print()
print("🎯 损失函数特性：")
print("  • 对自信但错误的预测进行重罚")
print("  • 当模型判断错误时提供强梯度")
print("  • 与 softmax 激活函数自然结合")
print("  • 通过 epsilon 裁剪保证数值稳定")
print()
print("💡 训练洞察：")
print("  损失函数通过提供梯度来指导学习")
print("  使模型朝正确预测方向更新")

✅ 交叉熵损失已实现！

🎯 损失函数特性：
  • 对自信但错误的预测进行重罚
  • 当模型判断错误时提供强梯度
  • 与 softmax 激活函数自然结合
  • 通过 epsilon 裁剪保证数值稳定

💡 训练洞察：
  损失函数通过提供梯度来指导学习
  使模型朝正确预测方向更新


## 🎵 唤醒词检测网络实现

### 网络结构：4 → 3 → 2

我们的唤醒词检测网络采用了精心设计的结构：

**输入层（4 个特征）：**
- 频谱特征（主频、频谱重心）
- 能量特征（RMS 能量、过零率）
- 时间特征（持续时间、停顿检测）
- 幅度特征（峰值幅度、动态范围）

**隐藏层（3 个神经元）：**
- 学习复杂特征组合
- 使用 Tanh 激活以获得平滑梯度
- 具有足够容量来处理简单的唤醒词模式

**输出层（2 个类别）：**
- 类别 0：检测到唤醒词
- 类别 1：未检测到唤醒词（背景/静音）
- 使用 Softmax 激活输出概率分布

### 嵌入式系统中的内存分析

这种结构专门为微控制器部署而设计：
- **总参数量**： (4×3 + 3) + (3×2 + 2) = 23 个参数
- **权重内存**：23 × 4 bytes = 92 bytes
- **激活内存**：推理时约 40 bytes
- **总推理内存**：<150 bytes（非常适合嵌入式设备！）

### 为什么是这个规模？

- **计算约束**：必须在微控制器上实时运行
- **内存约束**：需适应可用 RAM 的千字节级别
- **功耗约束**：尽量降低电池设备上的能耗
- **准确率约束**：仍需保持合理的唤醒词检测性能

In [6]:
# 唤醒词检测网络设置
print("🎵 唤醒词检测网络")
print("=" * 50)
print()
print("🏗️ 网络结构: 4 → 3 → 2")
print("  • 输入: 4 个音频特征")
print("  • 隐藏层: 3 个神经元，使用 tanh 激活")
print("  • 输出: 2 个类别（wake_word, no_wake_word)")
print()

# 测试用输入数据
audio_features = [0.2, -0.1, 0.5, 0.3]  # 模拟音频特征
target = [1, 0]  # 唤醒词检测（one-hot 编码）

print("📊 测试数据:")
print(f"  音频特征: {audio_features}")
print(f"    特征 0: {audio_features[0]}（频谱重心）")
print(f"    特征 1: {audio_features[1]}（过零率）")
print(f"    特征 2: {audio_features[2]}（RMS 能量）")
print(f"    特征 3: {audio_features[3]}（频谱滚降）")
print()
print(f"  目标: {target}")
print(f"    含义: 检测到唤醒词（类别 0)")
print()

# 创建输入和目标节点
print("🔧 创建计算图节点...")
input_node = ComputationNode(audio_features, name="audio_input", requires_grad=False)
target_node = ComputationNode(target, name="target", requires_grad=False)

print("✅ 输入节点已创建!")
print(f"  输入节点: {input_node}")
print(f"  目标节点: {target_node}")

🎵 唤醒词检测网络

🏗️ 网络结构: 4 → 3 → 2
  • 输入: 4 个音频特征
  • 隐藏层: 3 个神经元，使用 tanh 激活
  • 输出: 2 个类别（wake_word, no_wake_word)

📊 测试数据:
  音频特征: [0.2, -0.1, 0.5, 0.3]
    特征 0: 0.2（频谱重心）
    特征 1: -0.1（过零率）
    特征 2: 0.5（RMS 能量）
    特征 3: 0.3（频谱滚降）

  目标: [1, 0]
    含义: 检测到唤醒词（类别 0)

🔧 创建计算图节点...
📦 已创建节点 'audio_input'：
    形状: (4,)
    值: [ 0.2 -0.1  0.5  0.3]
    是否需要梯度: False
📦 已创建节点 'target'：
    形状: (2,)
    值: [1. 0.]
    是否需要梯度: False
✅ 输入节点已创建!
  输入节点: Node('audio_input', shape=(4,), grad_norm=0.000000)
  目标节点: Node('target', shape=(2,), grad_norm=0.000000)


In [7]:
# 第 1 层：音频特征 → 隐藏层（4 → 3）
print("\n🏗️ 构建第 1 层：音频特征 → 隐藏层")
print("-" * 55)

# 初始化第 1 层的权重和偏置
# 小随机权重可避免对称性问题
np.random.seed(42)  # 保证可复现结果
W1 = np.random.randn(3, 4).astype(np.float32) * 0.1  # 小初始化
b1 = np.zeros(3, dtype=np.float32)  # 偏置初始为 0

print("⚙️ 第 1 层参数:")
print(f"  权重矩阵 W1 形状: {W1.shape}")
print(f"  W1 值:\n{W1}")
print(f"  偏置向量 b1: {b1}")
print()

# 创建参数节点
W1_node = ComputationNode(W1, name="W1", requires_grad=True)
b1_node = ComputationNode(b1, name="b1", requires_grad=True)

# 第 1 层前向传播
print("🔄 正在进行第 1 层前向传播...")
z1_node = linear_layer(input_node, W1_node, b1_node, name="layer1_linear")
a1_node = tanh_activation(z1_node, name="layer1_activation")

print(f"✅ 第 1 层完成!")
print(f"  预激活值 (z1): {z1_node.value}")
print(f"  激活后值 (a1): {a1_node.value}")
print(f"  激活范围: [{np.min(a1_node.value):.3f}, {np.max(a1_node.value):.3f}]")


🏗️ 构建第 1 层：音频特征 → 隐藏层
-------------------------------------------------------
⚙️ 第 1 层参数:
  权重矩阵 W1 形状: (3, 4)
  W1 值:
[[ 0.04967142 -0.01382643  0.06476886  0.15230298]
 [-0.02341534 -0.0234137   0.15792128  0.07674348]
 [-0.04694744  0.054256   -0.04634177 -0.04657298]]
  偏置向量 b1: [0. 0. 0.]

📦 已创建节点 'W1'：
    形状: (3, 4)
    值: [[ 0.04967142 -0.01382643  0.06476886  0.15230298]
 [-0.02341534 -0.0234137   0.15792128  0.07674348]
 [-0.04694744  0.054256   -0.04634177 -0.04657298]]
    是否需要梯度: True
📦 已创建节点 'b1'：
    形状: (3,)
    值: [0. 0. 0.]
    是否需要梯度: True
🔄 正在进行第 1 层前向传播...

🔄 计算 layer1_linear 层
    输入形状: (4,)
    权重形状: (3, 4)
    偏置形状: (3,)
    矩阵乘法后: (3,)
    加偏置后: (3,)
    输出值: [ 0.08939224  0.09964199 -0.05195787]
📦 已创建节点 'layer1_linear_output'：
    形状: (3,)
    值: [ 0.08939224  0.09964199 -0.05195787]
    是否需要梯度: True

🎛️ 计算 layer1_activation 激活
    输入值: [ 0.08939224  0.09964199 -0.05195787]
📦 已创建节点 'layer1_activation_output'：
    形状: (3,)
    值: [ 0.08915489  0.09931352 -0.05

In [8]:
# 第 2 层：隐藏层 → 输出层（3 → 2）
print("\n🏗️ 构建第 2 层：隐藏层 → 输出类别")
print("-" * 55)

# 初始化第 2 层的权重和偏置
W2 = np.random.randn(2, 3).astype(np.float32) * 0.1
b2 = np.zeros(2, dtype=np.float32)

print("⚙️ 第 2 层参数:")
print(f"  权重矩阵 W2 形状: {W2.shape}")
print(f"  W2 值:\n{W2}")
print(f"  偏置向量 b2: {b2}")
print()

# 创建参数节点
W2_node = ComputationNode(W2, name="W2", requires_grad=True)
b2_node = ComputationNode(b2, name="b2", requires_grad=True)

# 第 2 层前向传播
print("🔄 正在进行第 2 层前向传播...")
z2_node = linear_layer(a1_node, W2_node, b2_node, name="layer2_linear")
probs_node = softmax_activation(z2_node, name="layer2_softmax")

print(f"✅ 第 2 层完成!")
print(f"  Softmax 前 logits (z2): {z2_node.value}")
print(f"  最终概率: {probs_node.value}")
print(f"  预测类别: {np.argmax(probs_node.value)} ({'唤醒词' if np.argmax(probs_node.value) == 0 else '非唤醒词'})")
print(f"  置信度: {np.max(probs_node.value):.1%}")


🏗️ 构建第 2 层：隐藏层 → 输出类别
-------------------------------------------------------
⚙️ 第 2 层参数:
  权重矩阵 W2 形状: (2, 3)
  W2 值:
[[ 0.02419623 -0.19132803 -0.17249179]
 [-0.05622875 -0.10128311  0.03142473]]
  偏置向量 b2: [0. 0.]

📦 已创建节点 'W2'：
    形状: (2, 3)
    值: [[ 0.02419623 -0.19132803 -0.17249179]
 [-0.05622875 -0.10128311  0.03142473]]
    是否需要梯度: True
📦 已创建节点 'b2'：
    形状: (2,)
    值: [0. 0.]
    是否需要梯度: True
🔄 正在进行第 2 层前向传播...

🔄 计算 layer2_linear 层
    输入形状: (3,)
    权重形状: (2, 3)
    偏置形状: (2,)
    矩阵乘法后: (2,)
    加偏置后: (2,)
    输出值: [-0.00789    -0.01670315]
📦 已创建节点 'layer2_linear_output'：
    形状: (2,)
    值: [-0.00789    -0.01670315]
    是否需要梯度: True

🎯 计算 layer2_softmax 激活
    输入 logits: [-0.00789    -0.01670315]
    稳定化后: [ 0.         -0.00881315]
📦 已创建节点 'layer2_softmax_output'：
    形状: (2,)
    值: [0.5022033  0.49779674]
    是否需要梯度: True
    输出概率: [0.5022033  0.49779674]
    概率和: 1.000000（应为 1.0）
✅ 第 2 层完成!
  Softmax 前 logits (z2): [-0.00789    -0.01670315]
  最终概率: [0.5022033  0.49

In [9]:
# 计算损失并分析结果
print("\n🎯 计算损失与网络表现")
print("-" * 50)

# 计算交叉熵损失
loss_node = cross_entropy_loss(probs_node, target_node, name="training_loss")

print("\n📈 前向传播总结")
print("=" * 40)
show("网络前向传播结果",
     ("输入特征", input_node.value),
     ("隐藏层激活", a1_node.value),
     ("输出 logits", z2_node.value),
     ("输出概率", probs_node.value),
     ("训练损失", f"{loss_node.value:.6f}"),
     ("预测类别", np.argmax(probs_node.value)),
     ("真实类别", np.argmax(target_node.value)),
     ("预测是否正确?", np.argmax(probs_node.value) == np.argmax(target_node.value)))

# 网络解释
print(f"\n🔍 网络解释:")
print(f"  网络预测类别 {np.argmax(probs_node.value)}，置信度为 {np.max(probs_node.value):.1%}")
print(f"  真实类别为 {np.argmax(target_node.value)}")

if np.argmax(probs_node.value) == np.argmax(target_node.value):
    print(f"  ✅ 预测正确！损失 = {loss_node.value:.6f}")
else:
    print(f"  ❌ 预测错误。损失 = {loss_node.value:.6f}")
    print(f"  较高的损失表明模型需要更多训练")

print(f"\n💡 概率含义:")
for i, (prob, is_target) in enumerate(zip(probs_node.value, target_node.value)):
    class_name = "唤醒词" if i == 0 else "非唤醒词"
    status = "✅ 真类别" if is_target else "❌ 假类别"
    print(f"  类别 {i} ({class_name}): {prob:.1%} 置信度 {status}")


🎯 计算损失与网络表现
--------------------------------------------------

🎯 计算 training_loss 损失
    预测值: [0.5022033  0.49779674]
    目标值: [1. 0.]
    安全预测值（裁剪后）: [0.5022033  0.49779674]
    对数预测值: [-0.68875027 -0.6975634 ]
    损失项: [-0.68875027 -0.        ]
    最终损失: 0.6887502670288086
📦 已创建节点 'training_loss_output'：
    形状: ()
    值: 0.6887502670288086
    是否需要梯度: True

📈 前向传播总结
网络前向传播结果
  输入特征: [ 0.2 -0.1  0.5  0.3]
  隐藏层激活: [ 0.08915489  0.09931352 -0.05191116]
  输出 logits: [-0.00789    -0.01670315]
  输出概率: [0.5022033  0.49779674]
  训练损失: 0.688750
  预测类别: 0
  真实类别: 0
  预测是否正确?: True

🔍 网络解释:
  网络预测类别 0，置信度为 50.2%
  真实类别为 0
  ✅ 预测正确！损失 = 0.688750

💡 概率含义:
  类别 0 (唤醒词): 50.2% 置信度 ✅ 真类别
  类别 1 (非唤醒词): 49.8% 置信度 ❌ 假类别


## 💾 内存分析：嵌入式系统约束

### 内存预算分析

在我们的 1KB 限制下，我们需要仔细核算每一字节：

**参数存储：**
- 权重：W1 (3×4) + W2 (2×3) = 18 个 float32 值 = 72 bytes
- 偏置：b1 (3) + b2 (2) = 5 个 float32 值 = 20 bytes
- **总参数量：92 bytes**

**激活存储（前向传播）：**
- 输入：4 个 float32 = 16 bytes
- 隐藏层：3 个 float32 = 12 bytes
- 输出：2 个 float32 = 8 bytes
- **总激活量：36 bytes**

**梯度存储（反向传播）：**
- 与参数相同：92 bytes

**临时工作区：**
- 临时计算：约 50 bytes

**总计：约 270 bytes（远低于 1KB！）**

### 内存优化策略

1. **混合精度**：前向传播使用 float16，梯度使用 float32
2. **原地操作**：尽可能复用内存
3. **流式处理**：一次只处理一个样本（不使用批量）
4. **量化**：推理时使用更低精度

### 现实世界影响

这个分析表明：只要进行精心的内存管理，复杂神经网络也可以在微控制器上运行！

In [10]:
def analyze_memory_usage():
    """
    针对嵌入式部署的综合内存分析
    
    这个函数会计算我们唤醒词检测网络的精确内存需求，
    并确保其能满足嵌入式系统的资源限制。
    """
    print("💾 综合内存分析")
    print("=" * 45)
    print()
    
    # 收集计算图中的所有节点
    all_nodes = [
        input_node, target_node,
        W1_node, b1_node, z1_node, a1_node,
        W2_node, b2_node, z2_node, probs_node,
        loss_node
    ]
    
    print("📊 按节点拆分的内存占用：")
    print("节点名称           | 形状      | 元素数 | 值占用字节 | 梯度占用字节 | 总计")
    print("-" * 80)
    
    total_value_bytes = 0
    total_grad_bytes = 0
    
    for node in all_nodes:
        elements = node.value.size
        value_bytes = elements * 4  # float32 = 4 bytes
        grad_bytes = elements * 4 if node.requires_grad else 0
        total_bytes = value_bytes + grad_bytes
        
        total_value_bytes += value_bytes
        total_grad_bytes += grad_bytes
        
        print(f"{node.name:<18} | {str(node.value.shape):<10} | {elements:<8} | {value_bytes:<11} | {grad_bytes:<10} | {total_bytes}")
    
    print("-" * 80)
    print(f"{'TOTALS':<18} | {'':10} | {'':8} | {total_value_bytes:<11} | {total_grad_bytes:<10} | {total_value_bytes + total_grad_bytes}")
    
    # 分类汇总
    print(f"\n📈 内存分类汇总：")
    
    # 参数（权重和偏置）
    param_nodes = [W1_node, b1_node, W2_node, b2_node]
    param_bytes = sum(node.value.nbytes + (node.gradient.nbytes if node.requires_grad else 0) for node in param_nodes)
    
    # 激活值（中间结果）
    activation_nodes = [input_node, z1_node, a1_node, z2_node, probs_node]
    activation_bytes = sum(node.value.nbytes for node in activation_nodes)
    
    # 损失和目标值
    other_bytes = target_node.value.nbytes + loss_node.value.nbytes
    
    print(f"  参数（权重 + 偏置 + 梯度）：{param_bytes} bytes")
    print(f"  激活值（前向传播存储）：{activation_bytes} bytes")
    print(f"  损失和目标值：{other_bytes} bytes")
    print(f"  总计：{param_bytes + activation_bytes + other_bytes} bytes")
    
    # 约束分析
    constraint = 1024  # 1KB 约束
    total_used = total_value_bytes + total_grad_bytes
    
    print(f"\n🎯 约束分析：")
    print(f"  内存预算：{constraint} bytes (1 KB)")
    print(f"  已使用内存：{total_used} bytes")
    print(f"  剩余空间：{constraint - total_used} bytes")
    print(f"  内存占用率：{(total_used/constraint)*100:.1f}%")
    
    if total_used <= constraint:
        print(f"  ✅ 在预算内！系统可以部署到嵌入式设备")
    else:
        print(f"  ❌ 超出预算！需要对嵌入式部署进行优化")
    
    # 优化建议
    print(f"\n💡 内存优化机会：")
    if total_used <= constraint:
        print(f"  • 剩余 {constraint - total_used} bytes 可用于：")
        print(f"    - 更大的网络（更多隐藏单元）")
        print(f"    - 流式音频输入缓冲")
        print(f"    - 多个音频特征提取器")
    else:
        print(f"  • 使用 float16 作为激活值（可减少 50%）")
        print(f"  • 将梯度量化为 int16（可减少 50% 梯度存储）")
        print(f"  • 使用原地操作复用激活内存")
    
    return total_used

# 运行内存分析
memory_used = analyze_memory_usage()

💾 综合内存分析

📊 按节点拆分的内存占用：
节点名称           | 形状      | 元素数 | 值占用字节 | 梯度占用字节 | 总计
--------------------------------------------------------------------------------
audio_input        | (4,)       | 4        | 16          | 0          | 16
target             | (2,)       | 2        | 8           | 0          | 8
W1                 | (3, 4)     | 12       | 48          | 48         | 96
b1                 | (3,)       | 3        | 12          | 12         | 24
layer1_linear_output | (3,)       | 3        | 12          | 12         | 24
layer1_activation_output | (3,)       | 3        | 12          | 12         | 24
W2                 | (2, 3)     | 6        | 24          | 24         | 48
b2                 | (2,)       | 2        | 8           | 8          | 16
layer2_linear_output | (2,)       | 2        | 8           | 8          | 16
layer2_softmax_output | (2,)       | 2        | 8           | 8          | 16
training_loss_output | ()         | 1        | 4           | 4          | 8
----

## ⬅️ 反向模式自动微分：反向传播

### 理解反向传播

反向传播正是关键所在！从损失值（一个标量）出发，我们将梯度沿整个网络反向传播，计算每个参数应该变动多少。

### 反向传播算法

1. **初始化损失**：将损失梯度设为 1.0（∂Loss/∂Loss = 1）
2. **逆拓扑顺序**：按从输出到输入的顺序访问节点
3. **应用链式法则**：每个节点为其输入计算梯度
4. **累积梯度**：对多条路径贡献求和

### 为什么它有效

微积分中的链式法则：**∂Loss/∂param = ∂Loss/∂output × ∂output/∂param**

每个反向函数都实现了第二项，而反向传播则自动处理第一项的梯度传播。

### 计算效率

- **一次反向传播**可计算所有参数梯度
- **对神经网络特别高效**：参数很多，但只有一个损失值
- **扩展性很好**：适用于数百万参数网络

### 内存与计算的权衡

- **内存**：必须保存前向传播中的中间值
- **计算**：每个操作前向计算一次，反向计算一次
- **结果**：计算量约为 2 倍，但这使得复杂模型能够训练

In [11]:
print("⬅️ 反向模式自动微分")
print("=" * 50)
print()
print("🌱 从损失值开始反向传播...")
print("这里我们会计算整个网络中所有参数的梯度！")
print()

# 第 1 步：为反向传播设定初值
print("步骤 1：为反向传播设定初值")
print("-" * 35)
loss_node.gradient = np.array(1.0, dtype=np.float32)
print(f"✅ 已设置损失梯度 = {loss_node.gradient}")
print("   这表示：∂Loss/∂Loss = 1.0（数学事实）")
print()

# 第 2 步：定义反向传播执行顺序（逆拓扑顺序）
print("步骤 2：定义反向传播执行顺序")
print("-" * 45)
backward_nodes = [loss_node, probs_node, z2_node, a1_node, z1_node]

print("📋 反向传播执行顺序：")
for i, node in enumerate(backward_nodes):
    print(f"  {i+1}. {node.name}（将梯度传递给它的输入）")
print()
print("💡 这个顺序确保梯度能够在图中正确流动")
print("   每个节点都在计算自己的输入梯度之前先接收梯度")
print()

# 第 3 步：执行反向传播
print("步骤 3：执行反向传播")
print("-" * 35)
print("🔄 正在通过反向模式自动微分计算梯度...")
print()

for i, node in enumerate(backward_nodes):
    if node.backward_fn:
        print(f"🔄 第 {i+1} 步：处理 {node.name}")
        print(f"   当前梯度: {node.gradient}")
        node.backward_fn()
        print(f"   ✅ 梯度已计算并传播")
        print()

print("🎉 反向传播完成！")
print("所有参数梯度都已被自动计算完成！")

⬅️ 反向模式自动微分

🌱 从损失值开始反向传播...
这里我们会计算整个网络中所有参数的梯度！

步骤 1：为反向传播设定初值
-----------------------------------
✅ 已设置损失梯度 = 1.0
   这表示：∂Loss/∂Loss = 1.0（数学事实）

步骤 2：定义反向传播执行顺序
---------------------------------------------
📋 反向传播执行顺序：
  1. training_loss_output（将梯度传递给它的输入）
  2. layer2_softmax_output（将梯度传递给它的输入）
  3. layer2_linear_output（将梯度传递给它的输入）
  4. layer1_activation_output（将梯度传递给它的输入）
  5. layer1_linear_output（将梯度传递给它的输入）

💡 这个顺序确保梯度能够在图中正确流动
   每个节点都在计算自己的输入梯度之前先接收梯度

步骤 3：执行反向传播
-----------------------------------
🔄 正在通过反向模式自动微分计算梯度...

🔄 第 1 步：处理 training_loss_output
   当前梯度: 1.0

⬅️ training_loss 的反向传播
    对预测值的梯度: [-1.9912255 -0.       ]
    梯度解释:
      类别 0（TRUE）：pred=0.5022, grad=-1.9912
        → 强梯度推动概率上升
      类别 1（FALSE）：pred=0.4978, grad=-0.0000
        → 弱梯度推动概率下降
   ✅ 梯度已计算并传播

🔄 第 2 步：处理 layer2_softmax_output
   当前梯度: [-1.9912255  0.       ]

⬅️ layer2_softmax 的反向传播
    Jacobian 形状: (2, 2)
    输入梯度: [-0.49779668  0.4977967 ]
   ✅ 梯度已计算并传播

🔄 第 3 步：处理 layer2_linear_output
   当前梯

In [12]:
print("\n📊 梯度分析与解读")
print("=" * 50)
print()

def analyze_gradients():
    """
    分析计算得到的梯度，理解网络学到了什么，
    以及参数应该如何更新以提升性能。
    """
    
    print("🔍 参数梯度摘要：")
    print("-" * 40)
    
    # 收集所有参数梯度
    param_gradients = {
        'W1': W1_node.gradient,
        'b1': b1_node.gradient,
        'W2': W2_node.gradient,
        'b2': b2_node.gradient
    }
    
    for name, grad in param_gradients.items():
        grad_norm = np.linalg.norm(grad)
        grad_max = np.max(np.abs(grad))
        grad_mean = np.mean(grad)
        
        print(f"📈 {name} 梯度：")
        print(f"    形状: {grad.shape}")
        print(f"    值: {grad}")
        print(f"    范数（大小）: {grad_norm:.6f}")
        print(f"    最大绝对值: {grad_max:.6f}")
        print(f"    均值: {grad_mean:.6f}")
        print()
    
    # 解读梯度含义
    print("🧠 梯度解读：")
    print("-" * 30)
    
    print("💡 这些梯度告诉我们：")
    print()
    
    # 分析 W2 梯度（输出层）
    print(f"🎯 输出层 (W2) 分析：")
    print(f"   W2 连接隐藏层与输出类别")
    for i in range(W2_node.gradient.shape[0]):
        class_name = "唤醒词" if i == 0 else "非唤醒词"
        row_grad = W2_node.gradient[i]
        direction = "增加" if np.mean(row_grad) > 0 else "减少"
        strength = "强" if np.linalg.norm(row_grad) > 0.1 else "弱"
        
        print(f"   类别 {i} ({class_name}): {strength} {direction} 信号")
        print(f"     梯度: {row_grad}")
    print()
    
    # 分析梯度流强度
    total_grad_norm = sum(np.linalg.norm(grad) for grad in param_gradients.values())
    print(f"📊 梯度流分析：")
    print(f"   总梯度大小: {total_grad_norm:.6f}")
    
    if total_grad_norm > 1.0:
        print(f"   🚨 检测到较大梯度——考虑梯度裁剪")
    elif total_grad_norm < 0.001:
        print(f"   ⚠️ 检测到很小的梯度——学习可能会比较缓慢")
    else:
        print(f"   ✅ 梯度大小适中，适合学习")
    
    return param_gradients

# 运行梯度分析
gradients = analyze_gradients()

# 展示学习方向
print("🎓 学习方向分析：")
print("-" * 35)
print("如果应用这些梯度（梯度下降），网络会：")

learning_rate = 0.01
print(f"学习率 = {learning_rate}:")

for name, grad in gradients.items():
    update_magnitude = learning_rate * np.linalg.norm(grad)
    print(f"  {name}: 更新量 = {update_magnitude:.6f}")
    
    if 'W' in name:  # 权重矩阵
        print(f"    → 权重连接将被调整")
    else:  # 偏置向量
        print(f"    → 偏置项将被移动")

print()
print("💡 该网络正在学习：")
print("   • 强化有助于正确分类唤醒词的连接")
print("   • 削弱导致错误分类的连接")
print("   • 调整偏置来优化决策边界")


📊 梯度分析与解读

🔍 参数梯度摘要：
----------------------------------------
📈 W1 梯度：
    形状: (3, 4)
    值: [[-0.00794341  0.00397171 -0.01985853 -0.01191512]
 [ 0.00887639 -0.0044382   0.02219098  0.01331459]
 [ 0.02024709 -0.01012354  0.05061771  0.03037063]]
    范数（大小）: 0.073351
    最大绝对值: 0.050618
    均值: 0.007943

📈 b1 梯度：
    形状: (3,)
    值: [-0.03971707  0.04438195  0.10123542]
    范数（大小）: 0.117456
    最大绝对值: 0.101235
    均值: 0.035300

📈 W2 梯度：
    形状: (2, 3)
    值: [[-0.04438101 -0.04943794  0.0258412 ]
 [ 0.04438101  0.04943794 -0.02584121]]
    范数（大小）: 0.100812
    最大绝对值: 0.049438
    均值: 0.000000

📈 b2 梯度：
    形状: (2,)
    值: [-0.49779668  0.4977967 ]
    范数（大小）: 0.703991
    最大绝对值: 0.497797
    均值: 0.000000

🧠 梯度解读：
------------------------------
💡 这些梯度告诉我们：

🎯 输出层 (W2) 分析：
   W2 连接隐藏层与输出类别
   类别 0 (唤醒词): 弱 减少 信号
     梯度: [-0.04438101 -0.04943794  0.0258412 ]
   类别 1 (非唤醒词): 弱 增加 信号
     梯度: [ 0.04438101  0.04943794 -0.02584121]

📊 梯度流分析：
   总梯度大小: 0.995610
   ✅ 梯度大小适中，适合学习
🎓 学习方向分析：
---

## 🎉 练习 2 总结：关键成果

### 我们构建了什么

✅ **完整的计算图实现**
- 基于节点的自动微分架构
- 前向传播与策略性中间值存储
- 反向传播与自动梯度计算

✅ **真实的唤醒词检测网络**
- 4 个音频特征 → 3 个隐藏层 → 2 个类别架构
- 合适的激活函数（tanh、softmax）
- 用于分类的交叉熵损失

✅ **内存高效设计**
- 总内存使用约为 270 bytes，远低于 1KB 约束
- 仅存储反向传播真正需要的关键值
- 可直接部署到嵌入式系统

✅ **教学型实现**
- 每一步运算都附有详细说明
- 数学原理讲解清晰
- 结合真实世界场景与应用背景

### 我们学到的关键见解

🧠 **计算图非常强大**
- 能够为任意复杂函数自动计算梯度
- 是现代机器学习框架（PyTorch、TensorFlow）的基础
- 非常适合实现神经网络抽象

🔄 **反向模式自动微分高效**
- 一次反向传播即可计算所有参数梯度
- 能很好地扩展到大量参数规模
- 对训练大型神经网络至关重要

💾 **内存管理至关重要**
- 嵌入式系统需要谨慎分析内存使用
- 需要在内存占用和计算量之间做出平衡
- 策略性检查点保存使更大模型成为可能

### 现实世界应用

这个实现展示了以下技术的核心原理：
- **智能音箱**（Alexa、Google Home、Siri）
- **移动端语音助手**
- **带语音控制的边缘 AI 设备**
- **保护隐私的联邦学习系统**

### 下一步

这个计算图为以下内容奠定了基础：
- 训练更大、更复杂的模型
- 实现更高级的优化算法
- 在真实嵌入式硬件上部署
- 参与联邦学习网络